In [1]:
import nest_asyncio 
nest_asyncio.apply()

import asyncio
import os
import fnmatch
from pathlib import Path
from ollama import AsyncClient


In [2]:


# 1. System Prompt targeting Qwen Coder for file analysis
SYSTEM_PROMPT = """
Role: Senior Java/Groovy & Grails Monolith Systems Architect
Task: Analyze a hybrid Grails 3.x and Spring Boot monolithic application for structural layout, caching architecture, data flow bottlenecks, and Redis integration.

[CONTEXT OVERVIEW]
The target application is a monolithic system built using Grails 3.x (operating on top of a Spring Boot container). It uses GORM for relational persistence alongside Redis for caching or transient data structures.

[INSTRUCTIONS FOR ANALYSIS EXECUTIONS]
Parse the code through the lens of a Grails 3.x + Spring Boot structural hierarchy.
NOTE: If this is a chunked file, analyze it as a partial snippet of a larger class context.

[OUTPUT FORMAT REQUIREMENT]
Output your assessment exactly in this structured format for the code provided:

### 📄 File: {filepath_placeholder}
#### 🏛️ Core Artifact Mapping
- **Component Path:** [Path]
- **Component Type:** [Grails Service / Spring Component / GORM Domain / Controller]
- **Lifecycle Management:** [Grails Lifecycle Managed / Spring Boot Container Managed]

#### 🔄 Dependencies & Data Flow
- **Inbound Injections:** [What beans/services are injected into this class]
- **Outbound Operations:** [What databases, services, or Redis keys does this class touch]

#### ⚡ Caching & Redis Footprint
- **Caching Mechanism:** [None / @Cacheable / Direct RedisTemplate / GORM Redis]
- **Key Strategy:** [Provide the Redis key pattern used, e.g., 'user:profile:status:\\$id']
- **Potential Failure Modes:** [List serialization errors, connection leaks, or GORM lazy-load issues]

---
"""


In [3]:
# 2. Aggregator Prompt for final summary generation
SUMMARY_PROMPT = """
Role: Data Systems Auditor
Task: Analyze the provided code evaluation text and generate a structured aggregate catalog summarizing all Redis cache configurations.

Instructions:
Extract all identified Redis cache keys, prefixes, data types, namespaces, and vulnerabilities from the text.

Output exactly in this format at the beginning of the file:
## 📊 Enterprise Redis Cache Catalog Summary
### 🔑 Identified Keys & Namespaces

| Namespace / Prefix | Estimated Type | Originating Component | Eviction Trigger |
| :--- | :--- | :--- | :--- |
| [e.g., `user:session:`] | [String/Hash] | [AuthService] | [Logout / Session Timeout] |

### ⚡ Caching Vulnerabilities & Failure Risks
- [List critical architectural issues like cross-container serialization gaps, un-annotated transactional boundaries, or tight GORM lazy-loading hooks here]
"""


In [4]:
# Config targets & limits
TARGET_EXTENSIONS = {'.groovy', '.java', '.yml', '.properties'}
EXCLUDE_DIRS = {'build', '.gradle', '.git', 'node_modules', '.gsp'}
  


In [5]:
import asyncio
from pathlib import Path
from ollama import AsyncClient  # Assumed client source based on syntax

# Define missing globals
MAX_CHARS_PER_CHUNK = 4000
SYSTEM_PROMPT = "You are an expert code reviewer. Analyze the following code snippet carefully."


def split_code_file(code_content: str, max_chars: int = MAX_CHARS_PER_CHUNK) -> list[str]:
    """Syntax-aware file-splitter component separating blocks at top-level brace boundaries."""
    lines = code_content.splitlines()
    chunks = []
    current_chunk = []
    current_length = 0
    brace_count = 0
    
    for line in lines:
        stripped = line.strip()
        current_chunk.append(line)
        current_length += len(line) + 1
        
        brace_count += stripped.count('{')
        brace_count -= stripped.count('}')
        
        # Ensure that only split if brace_count is safely back to 0 (fully closed scopes)
       
        if current_length >= max_chars and brace_count <= 0:
            chunks.append("\n".join(current_chunk))
            current_chunk = []
            current_length = 0
            brace_count = 0  # Reset brace count for the next chunk
            
    if current_chunk:
        chunks.append("\n".join(current_chunk))
        
    return chunks

async def analyze_file(client: AsyncClient, model_name: str, file_path: Path, output_file: Path):
    """Reads, chunks, and streams code analysis from Qwen Coder directly into a report file."""
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            code_content = f.read()
        
        if not code_content.strip():
            return

        code_chunks = split_code_file(code_content)
        total_chunks = len(code_chunks)
        
        try:
            relative_path = file_path.relative_to(Path.cwd())
        except ValueError:
            relative_path = file_path.name

        for index, chunk in enumerate(code_chunks, start=1):
            print(f"Processing: {relative_path} [{index}/{total_chunks}]...")

            user_message = (
                f"Project File: {file_path}\n"
                f"Chunk Status: Part {index} of {total_chunks}\n\n"
                f"Code Snippet:\n```\n{chunk}\n```"
            )

            # Fire up the async streaming request before opening the file
            response_stream = await client.chat(
                model=model_name,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_message}
                ],
                stream=True
            )

            # Open the file once per chunk and pull from the async iterator safely
            with open(output_file, 'a', encoding='utf-8') as out_f:
                out_f.write(f"\n## Analysis for `{relative_path}` (Part {index}/{total_chunks})\n")
        
                async for chunk_response in response_stream:
                    
                    # Adaptive safety: works whether response uses dict or attribute access:
                
                    if hasattr(chunk_response, 'message'):            
                        content = chunk_response.message.content
                    else:
                        content = chunk_response['message']['content']   
                    
                    # with open(output_file, 'a', encoding='utf-8') as out_f:
                    out_f.write(content)
            
                # with open(output_file, 'a', encoding='utf-8') as out_f:
                out_f.write("\n\n---\n")
            
            if total_chunks > 1:
                await asyncio.sleep(0.2)

    except Exception as e:
        print(f"❌ Error analyzing {file_path}: {e}")

In [6]:

from urllib import response


async def generate_redis_summary(client: AsyncClient, model_name: str, output_file: Path):
    """Reads the raw compiled report, requests an aggregated Redis summary block, and prepends it."""
    print("📊 Compiling Redis Cache Catalog summary block...")
    try:
        with open(output_file, 'r', encoding='utf-8') as f:
            compiled_report_data = f.read()

        response = await client.chat(
            model=model_name,
            messages=[
                {'role': 'system', 'content': SUMMARY_PROMPT},
                {'role': 'user', 'content': f"Generate the catalog based on this analysis text:\n\n{compiled_report_data}"}
            ]
        )
        
        if hasattr(response, 'message'):
            summary_content = response.message.content
        else:
            summary_content = response['message']['content']


        # Prepend the generated summary to the top of the file layout
        with open(output_file, 'r', encoding='utf-8') as f:
            original_content = f.read()

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("# Monolithic Application Code Structure Report\n")
            f.write(f"**Framework Matrix:** Grails 3.x / Spring Boot / GORM / Redis\n\n")
            f.write(summary_content)
            f.write("\n\n# Detailed File Breakdown\n")
            f.write(original_content.replace("# Monolithic Application Code Structure Report\n**Framework Matrix:** Grails 3.x / Spring Boot / GORM / Redis\n\n", ""))

        print("✨ Redis Cache Catalog summary successfully generated at the top of the report.")

    except Exception as e:
        print(f"❌ Error generating summary section: {e}")

In [7]:
import json



async def generate_suggestions(conversation_history: list) -> list:
    """
    Uses a small local Ollama model to dynamically predict the user's next choices.
    """
    # Use a tiny, fast model so the UX stays highly responsive
    model_name = "qwen2.5-coder:1.5b" 
    
    system_prompt = (
        "You are a UI optimization backend. Analyze the conversation history.\n"
        "Predict 3 distinct, highly relevant, short, and actionable follow-up questions or commands "
        "the user might want to click next.\n"
        "Requirements:\n"
        "- Length: 3 to 7 words per suggestion.\n"
        "- Format: Return ONLY a valid JSON array of strings.\n"
        "Example: [\"Explain the error code\", \"Show me the solution\", \"Try a different approach\"]"
    )
    
    # Clone history to avoid modifying the main conversation array
    messages = [{"role": "system", "content": system_prompt}]
    
    # Append the last couple of turns for context to keep context windows small
    messages.extend(conversation_history[-4:]) 
    
    try:
        # Call Ollama asynchronously, forcing JSON output
        response = await AsyncClient().chat(
            model=model_name,
            messages=messages,
            format="json",
            options={"temperature": 0.5} # Low temperature for more predictable structured outputs
        )
        
        # Parse and return the generated array
        suggestions = json.loads(response['message']['content'])
        return suggestions if isinstance(suggestions, list) else []
        
    except Exception as e:
        print(f"Failed to generate suggestion chips: {e}")
        # Fail gracefully with safe fallback suggestions
        return ["Tell me more", "Give me an example", "What are my next steps?"]


In [8]:
async def handle_user_interaction(user_prompt: str, chat_history: list):
    # 1. Append user message to history
    chat_history.append({"role": "user", "content": user_prompt})
    
    # 2. Let your main agent stream or generate its response
    print("Agent responding...")
    agent_output = ""
    
    # Simulation of your agent generation (e.g., via a larger Ollama model like qwen2.5-coder:14b)
    async for chunk in await AsyncClient().chat(model="qwen2.5-coder:14b", messages=chat_history, stream=True):
        content = chunk['message']['content']
        agent_output += content
        print(content, end="", flush=True) # Stream to UI terminal
    print("\n")
    
    # 3. Save agent output to history
    chat_history.append({"role": "assistant", "content": agent_output})
    
    # 4. Generate the suggestion boxes for the NEXT turn
    print("✨ Generating suggestion boxes...")
    chips = await generate_suggestions(chat_history)
    
    return {
        "agent_response": agent_output,
        "suggestion_chips": chips
    }




In [9]:
async def main():
    MODEL_NAME = "qwen2.5-coder:14b"
    EXCLUDE_DIRS = {"build", ".gradle", ".git", "node_modules", ".gsp", "bin", "out"}
    TARGET_EXTENSIONS = {".groovy", ".java", ".yml", ".properties"}
    EXCLUDE_FILE_PATTERNS = {"messages*.properties", "markdown_report.md"}

    EXCLUDE_PATH_KEYWORDS = {
        f"{os.sep}grails-app{os.sep}domain{os.sep}",
        f"{os.sep}domain{os.sep}",
    }

    client = AsyncClient()
    project_root = Path.cwd()
    output_report = project_root / "markdown_report.md"

    with open(output_report, "w", encoding="utf-8") as f:
        f.write("# Monolithic Application Code Structure Report\n")
        f.write("**Framework Matrix:** Grails 3.x / Spring Boot / GORM / Redis\n\n")

    print("🚀 Starting Monolith Structure Scan.")
    print(f"📝 Output pipeline targeting: {output_report.name}")

    files_to_analyze = []
    initial_path_to_analyze = ["/Users/cherylhoekstra/projects/grails-demo/grails-app"]

    for p in initial_path_to_analyze:
        path_obj = Path(p)
        if path_obj.is_file():
            files_to_analyze.append(path_obj)
        elif path_obj.is_dir():
            print(f"📂 Scanning directory: {p}")

            for root, dirs, files in os.walk(p):
                dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]

                for file_name in files:
                    file_path = Path(root) / file_name
                    is_excluded_by_name = any(
                        fnmatch.fnmatch(file_name, pattern)
                        for pattern in EXCLUDE_FILE_PATTERNS
                    )

                    normalized_path = str(file_path).replace("\\", "/")
                    is_excluded_by_path = any(
                        keyword in normalized_path for keyword in EXCLUDE_PATH_KEYWORDS
                    )

                    if (
                        file_path.suffix.lower() in TARGET_EXTENSIONS
                        and not is_excluded_by_name
                        and not is_excluded_by_path
                    ):
                        print(f"Appending file: {file_path}")
                        files_to_analyze.append(file_path)

    if not files_to_analyze:
        print("⚠️ No valid source files found matching targets.")
        return

    print(f"📋 Found {len(files_to_analyze)} files. Processing pipeline running...")

    for file_path in files_to_analyze:
        await analyze_file(client, MODEL_NAME, file_path, output_report)
        await asyncio.sleep(0.5)

    await generate_redis_summary(client, MODEL_NAME, output_report)
    print(f"\n✨ Success! Report written to `{output_report.name}`")

    history = []
    result = await handle_user_interaction("How do I parse a JSON string in Python?", history)

    print("\n--- UI Suggestion Chips Prepared ---")
    print(result["suggestion_chips"])


if __name__ == "__main__":
    asyncio.run(main())

🚀 Starting Monolith Structure Scan.
📝 Output pipeline targeting: markdown_report.md
📂 Scanning directory: /Users/cherylhoekstra/projects/grails-demo/grails-app
Appending file: /Users/cherylhoekstra/projects/grails-demo/grails-app/init/com/example/Application.groovy
Appending file: /Users/cherylhoekstra/projects/grails-demo/grails-app/init/com/example/BootStrap.groovy
Appending file: /Users/cherylhoekstra/projects/grails-demo/grails-app/controllers/com/example/UrlMappings.groovy
Appending file: /Users/cherylhoekstra/projects/grails-demo/grails-app/controllers/com/example/BookController.groovy
Appending file: /Users/cherylhoekstra/projects/grails-demo/grails-app/conf/application.yml
Appending file: /Users/cherylhoekstra/projects/grails-demo/grails-app/conf/application-development.yml
Appending file: /Users/cherylhoekstra/projects/grails-demo/grails-app/conf/spring/resources.groovy
📋 Found 7 files. Processing pipeline running...
Processing: Application.groovy [1/1]...
Processing: BootStra